# 05 - SFT 微调对比: 手写 SFTTrainer vs HF Trainer

对比 from-scratch SFTTrainer (手写训练循环) 与 HuggingFace Trainer 的 SFT 实现差异。

| 维度 | from-scratch | HuggingFace |
|------|-------------|-------------|
| 训练类 | `SFTTrainer(BaseTrainer)` | `Trainer` + `SFTDataCollator` |
| Loss Mask | 手动 `labels[:prompt_len] = -100` | `load_sft_dataset` 预处理 |
| 预训练权重 | `load_checkpoint()` + torch.load | `from_pretrained()` |
| 训练循环 | epoch + micro-batch 手写 | `Trainer.train()` 自动 |
| LoRA | `apply_lora()` + `merge_lora()` | PEFT `get_peft_model()` |

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import tempfile
from pathlib import Path

from data import ClearMindTokenizer, load_sft_dataset

## 0. 准备测试数据和 Tokenizer

In [ ]:
# 准备 tokenizer
tokenizer_path = '../outputs/tokenizer'
if Path(tokenizer_path).exists():
    tokenizer = ClearMindTokenizer.load(tokenizer_path)
else:
    tmpdir = tempfile.mkdtemp()
    corpus = Path(tmpdir) / 'corpus.txt'
    corpus.write_text('\n'.join([
        '深度学习是机器学习的一个分支。', '自然语言处理是人工智能的重要领域。',
        'Transformer uses self-attention mechanisms.', '预训练语言模型学习数据表示。',
    ] * 10))
    tokenizer = ClearMindTokenizer.train(str(corpus), vocab_size=500)

# 创建 SFT 测试数据
tmpdir = tempfile.mkdtemp()
sft_path = Path(tmpdir) / 'sft.jsonl'
with open(sft_path, 'w') as f:
    for item in [
        {'instruction': '什么是深度学习？', 'input': '', 'output': '深度学习是机器学习的一个分支。'},
        {'instruction': '解释 Transformer', 'input': '', 'output': 'Transformer 使用自注意力机制。'},
        {'instruction': '翻译', 'input': '深度学习', 'output': 'Deep learning'},
    ]:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Tokenizer vocab_size: {tokenizer.vocab_size}')

## 1. SFT 数据格式化 — Chat Template vs 手动拼接

In [ ]:
# === HuggingFace 方式: apply_chat_template ===
messages = [
    {'role': 'user', 'content': '什么是深度学习？'},
    {'role': 'assistant', 'content': '深度学习是机器学习的一个分支。'},
]

# 完整对话
full = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(f'完整对话:\n{repr(full)}')

# Prompt 部分 (用于计算 loss mask 分界)
prompt = tokenizer.apply_chat_template(
    [messages[0]], tokenize=False, add_generation_prompt=True
)
print(f'\nPrompt 部分:\n{repr(prompt)}')

# 对比: from-scratch 硬编码拼接
# prompt_text = f"Human: {instruction}\nAssistant: "
# full_text = f"Human: {instruction}\nAssistant: {response}"

## 2. Loss Mask — 只在 Response 部分计算 Loss

In [ ]:
# === HuggingFace 方式: load_sft_dataset 预处理 labels ===
sft_ds = load_sft_dataset(str(sft_path), tokenizer, max_length=128)
sample = sft_ds['train'][0]

labels = sample['labels']
input_ids = sample['input_ids']

masked = sum(1 for l in labels if l == -100)
valid = sum(1 for l in labels if l != -100)
print(f'Labels 统计:')
print(f'  Masked (prompt, -100): {masked} tokens')
print(f'  Valid (response):      {valid} tokens')

# 可视化 mask
boundary = next((i for i, l in enumerate(labels) if l != -100), len(labels))
print(f'\nPrompt (0:{boundary}): 不计算 loss')
print(f'Response ({boundary}:{len(labels)}): 计算 loss')

# 对比: from-scratch 手动计算 prompt_len
# prompt_ids = tokenizer.encode(prompt_text, add_bos=True, add_eos=False)
# labels = full_ids.copy()
# labels[:len(prompt_ids)] = [-100] * len(prompt_ids)

## 3. SFT DataCollator — 动态 Padding

In [ ]:
# === HuggingFace 方式: SFTDataCollator 动态 padding ===
from training.sft import SFTDataCollator

collator = SFTDataCollator(pad_token_id=tokenizer.pad_token_id)
batch = collator([sft_ds['train'][i] for i in range(min(2, len(sft_ds['train'])))])

print(f'Batch 形状:')
print(f'  input_ids:      {batch["input_ids"].shape}')
print(f'  attention_mask:  {batch["attention_mask"].shape}')
print(f'  labels:          {batch["labels"].shape}')

# labels 的 padding 位置为 -100
pad_mask = (batch['input_ids'] == tokenizer.pad_token_id)
print(f'\nPad 位置 labels 全为 -100: {(batch["labels"][pad_mask] == -100).all().item()}')

# 对比: from-scratch 在 __getitem__ 中固定长度 padding
# pad_len = max_seq_len - len(full_ids)
# full_ids = full_ids + [pad_id] * pad_len
# labels = labels + [-100] * pad_len

## 4. SFT 训练流程对比

In [ ]:
print('=== from-scratch SFTTrainer ===')
print('''
# 1. 加载预训练权重
load_checkpoint(model, pretrained_path)

# 2. 可选: 应用 LoRA
apply_lora(model, rank=8, alpha=16)

# 3. 手写 epoch 循环
for epoch in range(epochs):
    for batch in train_loader:
        logits, loss, _ = model(input_ids, labels)  # labels 含 -100 mask
        scaled_loss = loss / gradient_accumulation
        scaled_loss.backward()
        
        if micro_count >= gradient_accumulation:
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
    
    # 验证 + Early Stopping
    val_loss = validate(model, val_loader)
    if early_stopping.should_stop(val_loss): break
''')

print('=== HuggingFace Trainer SFT ===')
print('''
# 1. 加载预训练模型
model = ClearMindForCausalLM.from_pretrained(pretrained_path)

# 2. 加载数据 (labels 已含 -100 mask)
datasets = load_sft_dataset(data_path, tokenizer, max_length=512)

# 3. 一行代码训练
trainer = Trainer(
    model=model,
    args=TrainingArguments(num_train_epochs=2, learning_rate=2e-5, ...),
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=SFTDataCollator(tokenizer.pad_token_id),
)
trainer.train()
trainer.save_model(output_dir)
''')

## 5. 从预训练到 SFT — 完整管线

In [ ]:
print('=== 完整管线 (HF 版) ===')
print('''
# Step 1: 预训练
python scripts/train.py --stage pretrain --config configs/tiny.yaml
# → outputs/pretrain/ (config.json + model.safetensors)

# Step 2: SFT 从预训练模型继续
python scripts/train.py --stage sft --config configs/tiny.yaml \\
    --resume outputs/pretrain
# → outputs/sft/ (config.json + model.safetensors)

# Step 3: 加载 SFT 模型推理
from model import ClearMindForCausalLM
model = ClearMindForCausalLM.from_pretrained("outputs/sft")
model.generate(input_ids, max_new_tokens=64)
''')

print('=== 对比: from-scratch 管线 ===')
print('''
# Step 1: 预训练
python scripts/train.py --stage pretrain --config configs/tiny.yaml
# → outputs/pretrain/final.pth

# Step 2: SFT (手动 load checkpoint)
python scripts/train.py --stage sft --resume outputs/pretrain/final.pth
# → outputs/sft/final.pth

# Step 3: 加载模型推理
model = GPT(config)
model.load_state_dict(torch.load("outputs/sft/final.pth"))
generate(model, tokenizer, prompt)
''')

## 总结

| 功能 | from-scratch | HuggingFace |
|------|-------------|-------------|
| 数据格式化 | 硬编码 `Human: ... Assistant: ...` | `apply_chat_template()` Jinja2 |
| Loss Mask | 手动计算 `prompt_len` | `load_sft_dataset` 预处理 |
| Padding | `__getitem__` 固定长度 | `SFTDataCollator` 动态 padding |
| 预训练加载 | `torch.load(final.pth)` | `from_pretrained(dir)` |
| 训练循环 | 手写 epoch + micro-batch | `Trainer.train()` |
| LoRA | `apply_lora()` + `merge_lora()` | PEFT `get_peft_model()` (阶段 8) |
| Checkpoint | `torch.save` (.pth) | `save_pretrained` (config.json + safetensors) |
| 代码量 | ~250 行 (SFTTrainer) | ~40 行 (run_sft) |

**核心收获:** SFT 的核心逻辑（loss mask、小学习率、epoch 训练）完全一致，
HuggingFace 通过标准化接口大幅减少代码量，同时通过 `from_pretrained` / `save_pretrained`
实现模型在管线各阶段间的无缝传递。